# 1.5 Compile、Fit、Evaluate、Predict

這份 Notebook 使用 Wine 多類別分類資料集，完整示範 Keras 的 `compile()`、`fit()`、`evaluate()`、`predict()`。

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

tf.keras.utils.set_random_seed(42)

## 1. 載入 Wine 資料集

Wine 是 sklearn 內建的多類別分類資料集。每筆資料是一款葡萄酒的化學分析數值，目標是判斷它屬於哪一個酒品類別。這份資料乾淨、類別明確、特徵數適中，適合示範完整訓練流程。

In [ ]:
data = load_wine(as_frame=True)
df = data.frame.copy()
df['target_name'] = df['target'].map(dict(enumerate(data.target_names)))
df.head()

## 2. 切分資料與標準化


In [ ]:
X = data.data.values.astype('float32')
y = data.target.values.astype('int64')

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

print('classes:', data.target_names)
print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)

## 3. 建立模型


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(len(data.target_names), activation='softmax')
])

model.summary()

## 4. Compile：指定訓練設定


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## 5. Fit：執行訓練


In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=80,
    batch_size=16,
    verbose=0
)

## 6. Evaluate：評估模型


In [ ]:
train_result = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)

pd.DataFrame([train_result, test_result], index=['train', 'test'])


## 7. Predict：預測新資料


In [ ]:
prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(prob, axis=1)

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=data.target_names, digits=4))

pd.DataFrame(prob[:5], columns=data.target_names).assign(
    actual=[data.target_names[i] for i in y_test[:5]],
    predicted=[data.target_names[i] for i in y_pred[:5]]
)


## 8. 觀察訓練曲線


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Workflow')
plt.show()


## 9. 小結

Keras 的標準流程是 `compile → fit → evaluate → predict`。熟悉這四步後，就能把同樣流程套到 DNN、CNN、RNN 與 Transformer 任務。
